### Ingesting the circuits file 

In [0]:
catalog = "formula1"
bronze_schema = "practice_class"
landing_path = "/Volumes/formula1/landing/files"

In [0]:
from pyspark.sql import functions as f
from pyspark.sql.types import *

class Ingestion:
    def __init__(self, catalog:str, bronze_schema:str, landing_path:str, spark):
        #loading the predefined values
        self.spark = spark
        self.catalog = catalog
        self.bronze_schema = bronze_schema
        self.landing_path = landing_path
# helpers functions
    def source(self, relative_path:str):
        return f"{self.landing_path}/{relative_path}"
    
    def target_table(self, name:str):
        return f"{self.catalog}.{self.bronze_schema}.{name}"
    
    def read (self, file_format, schema, source_path, multiline=False): # multiline is here only for the json file not for csv 
        reader = self.spark.read.format(file_format).schema(schema)
        if file_format == 'csv':
           reader = reader.option('header','true')
        if file_format == 'json' and multiline:
           reader = reader.option('multiLine', 'true')
        return reader.load(source_path) 
    def add_metadata(self, df):
        return (
            df.withColumn('ingest_timestamp', f.current_timestamp())
              .withColumn('source_path', f.col('_metadata.file_path'))
        ) 
    def write(self, df, table_name): 
        (
            df
            .write
            .format('delta')
            .mode('overwrite')
            .option("overwriteSchema", "true")
            .saveAsTable(table_name)
        )
    def ingest_circuits(self):
        schema = StructType([
            StructField("circuitId",   StringType(), True),
            StructField("url",         StringType(), True),
            StructField("circuitName", StringType(), True),
            StructField("lat",         DoubleType(), True),
            StructField("long",        DoubleType(), True),
            StructField("locality",    StringType(), True),
            StructField("country",     StringType(), True),
        ])
        df = self.read("csv", schema, self.source("circuits.csv"))
        df_final = self.add_metadata(df)
        self.write(df_final, self.target_table("circuits_practice"))

    def ingest_races(self):
     schema = StructType([
      StructField('season', IntegerType(), True),
      StructField('round', IntegerType(), True),
      StructField('url', StringType(), True),
      StructField('raceName',StringType(), True),
      StructField('date', DateType(), True),
      StructField('circuitId', StringType(), True)
     ])
     df = self.read("csv", schema, self.source("races.csv"))
     df_final = self.add_metadata(df)
     self.write(df_final, self.target_table("races_practice"))

    def ingest_constructors(self):
        schema = "constructorId STRING, name STRING, nationality STRING, url STRING"
        df = self.read('json', schema, self.source('constructors.json'))
        df_final = self.add_metadata(df)
        self.write(df_final,self.target_table('constructors_practice'))
    def ingest_drivers(self):
        name_schema = StructType([
               StructField('givenName', StringType(), True),
               StructField('familyName', StringType(), True)])
        schema = StructType([
               StructField('driverId', StringType(), True),
               StructField('name', name_schema),
               StructField('dateOfBirth', DateType(), True),
               StructField('nationality', StringType(), True),
               StructField('URL', StringType(), True)])
        df = self.read ('json',schema, self.source('drivers.json'))
        df_final = self.add_metadata(df)
        self.write(df_final,self.target_table('drivers_practice'))
    def ingest_results( self):
        schema = 'date DATE,raceName string, round int, season int, url string, constructorId string, driverId string, grid int, laps int, number int, points double, position int, positionText string, status string'
        df = self.read('json',schema, self.source('results')) 
        df_final = self.add_metadata(df)
        self.write(df_final, self.target_table('results_practice'))
    def ingest_sprints(self): 
        schema = ' date DATE,raceName string, round int, season int, url string, constructorId string, driverId string, grid int, laps int, number int, points double, position int, positionText string, status string'
        df = self.read('json', schema, self.source('sprints'), multiline=True)
        df_final = self.add_metadata(df)
        self.write(df_final, self.target_table('sprints_practice'))


In [0]:
ingester = Ingestion(catalog, bronze_schema, landing_path, spark)
ingester.ingest_circuits()
ingester.ingest_races()
ingester.ingest_constructors()
ingester.ingest_drivers()
ingester.ingest_results()
ingester.ingest_sprints()